# Parking Sign Detection - YOLO11 Training (v2 - Quality Focused)

Training a parking sign detector for North American street signs using YOLO11.

**Dataset:** ~8,600+ images, 1 class (parking_sign)
- Combined from: parking-sign-coco (Roboflow) + sf-parking-signs (Figure Eight)
- All images resized to 640x640 (up from 512)
- 10 sign-sized negative background crops per training image to reduce false positives
- Train/Val/Test split: 80/10/10

**Model:** YOLO11m (Medium - better discrimination than nano)

**Training:** 150 epochs, cosine LR decay, 2 GPUs (DDP), aggressive augmentation, multi-scale

## 1. Setup

In [ ]:
# Fix numpy version first (torchvision requires numpy<2)
!pip uninstall numpy -y -q
!pip install "numpy<2" -q

# Then install ultralytics
!pip install ultralytics -q

# Restart runtime after this cell if needed

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import yaml
import os

# Kaggle dataset path (nested directory structure confirmed by user)
# The dataset is mounted at /kaggle/input/parking-sign-detection-coco-dataset
# But inside that, there is ANOTHER folder with the same name.
DATASET_PATH = Path("/kaggle/input/parking-sign-detection-coco-dataset/parking-sign-detection-coco-dataset")
OUTPUT_PATH = Path("/kaggle/working")
DATA_YAML = DATASET_PATH / "data.yaml"

print(f"Dataset Path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

if DATASET_PATH.exists():
    print(f"Contents: {list(DATASET_PATH.iterdir())}")
    
    # Parse data.yaml
    with open(DATA_YAML) as f:
        data_config = yaml.safe_load(f)
    
    print("Original config:", data_config)
    
    # Update paths to use absolute paths since we are in a read-only input dir
    # and need to point strictly to the files.
    new_config = data_config.copy()
    
    # Fix train/val/test paths
    # The YAML likely contains relative paths like 'train/images' or '../train/images'
    # We force them to be absolute paths to the 'images' subdirectory
    for split in ["train", "val", "test"]:
        # Handle keys like 'valid' vs 'val'
        keys = [split]
        if split == 'val': keys.append('valid')
        
        for k in keys:
            if k in new_config:
                # Construct absolute path to the images directory
                # Structure is DATASET_PATH / <split_name> / images
                # Default split names in folder: train, valid, test
                folder_name = "valid" if split == "val" else split
                
                # Check if specific folder exists
                img_path = DATASET_PATH / folder_name / "images"
                if not img_path.exists():
                    # Try without 'images' subdir if flat
                    img_path = DATASET_PATH / folder_name
                
                if img_path.exists():
                    new_config[k] = str(img_path)
                    print(f"Resolved {k} to {img_path}")
                else:
                    print(f"WARNING: Could not find image path for {k} at {img_path}")

    # Set root path to avoid relative path processing issues
    new_config['path'] = str(DATASET_PATH)
    
    # Write corrected data.yaml to working directory (writable)
    FIXED_DATA_YAML = OUTPUT_PATH / "data.yaml"
    with open(FIXED_DATA_YAML, "w") as f:
        yaml.dump(new_config, f)
    
    DATA_YAML = FIXED_DATA_YAML
    print(f"\nFixed data.yaml written to {DATA_YAML}")
    !cat {DATA_YAML}
else:
    print("ERROR: DATASET_PATH not found. Please check directory structure in /kaggle/input")
    !ls -R /kaggle/input | head -n 20

## 2. Dataset Verification

In [ ]:
# Count images in each split
for split in ["train", "valid", "test"]:
    try:
        # Check flexible names for 'val'/'valid'
        folder_name = split
        if split == 'val' and not (DATASET_PATH / split).exists():
             folder_name = 'valid'
             
        img_dir = DATASET_PATH / folder_name / "images"
        label_dir = DATASET_PATH / folder_name / "labels"
        
        if img_dir.exists():
            n_images = len(list(img_dir.glob("*.jpg")))
            n_labels = len(list(label_dir.glob("*.txt")))
            print(f"{split} ({folder_name}): {n_images} images, {n_labels} labels")
        else:
            print(f"{split}: Directory not found at {img_dir}")
    except Exception as e:
        print(f"Error checking {split}: {e}")

## 3. Training Configuration

### Augmentation Strategy (v2 - Aggressive)
Previous baseline had high FP (cars, business signs) and missed small signs.
Now using full augmentation suite for maximum generalization:
- **Mosaic**: Combines 4 images, forces multi-scale learning
- **Scale ±50%**: Explicitly trains on varied sign sizes
- **HSV**: Color/brightness variation for lighting robustness
- **Geometric**: Rotation, shear for angle invariance
- **MixUp**: Blends images to reduce overconfident predictions

### Training Enhancements
- **Model**: YOLO11m (medium) for better feature discrimination
- **Resolution**: 640x640 (up from 512) for small sign detection
- **Extended training**: 150 epochs (patience 40)
- **Cosine LR decay**: Gradual learning rate reduction
- **Multi-GPU**: 2 GPUs with DDP for faster training
- **Negative samples**: Background crops in training set to learn what is NOT a sign

In [ ]:
# Training parameters - v2 quality-focused, optimized for 2 GPU setup
TRAIN_PARAMS = {
    "data": str(DATA_YAML),
    "epochs": 150,
    "imgsz": 640,
    "batch": 16,  # Reduced for larger model + larger images
    "workers": 4,
    "patience": 40,
    "save_period": 25,
    "project": str(OUTPUT_PATH / "runs"),
    "name": "parking_sign_detector_v2",
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
    # Multi-GPU (DDP)
    "device": "0,1",
    # Cosine LR decay
    "cos_lr": True,
    "lr0": 0.01,
    "lrf": 0.01,
    # Aggressive augmentation for quality
    "mosaic": 1.0,          # Combine 4 images - forces multi-scale learning
    "mixup": 0.15,          # Blend images - reduces overconfident FPs
    "copy_paste": 0.1,      # Copy-paste augmentation
    "degrees": 10.0,        # Rotation ±10°
    "translate": 0.2,       # Translation ±20%
    "scale": 0.5,           # Scale ±50% - critical for small sign detection
    "shear": 3.0,           # Shear ±3°
    "perspective": 0.0003,  # Slight perspective warp
    "fliplr": 0.5,          # Horizontal flip
    "flipud": 0.0,          # No vertical flip (signs have orientation)
    "hsv_h": 0.015,         # Hue shift
    "hsv_s": 0.7,           # Saturation shift
    "hsv_v": 0.4,           # Brightness shift
    # Close mosaic late for fine-tuning
    "close_mosaic": 20,     # Disable mosaic for last 20 epochs
}

print(f"Training config: {len(TRAIN_PARAMS)} parameters")
print(f"Model: yolo11m | Epochs: {TRAIN_PARAMS['epochs']}, Patience: {TRAIN_PARAMS['patience']}")
print(f"Image size: {TRAIN_PARAMS['imgsz']} | Device: {TRAIN_PARAMS['device']} (2 GPUs)")
print(f"Augmentation: mosaic={TRAIN_PARAMS['mosaic']}, scale={TRAIN_PARAMS['scale']}, mixup={TRAIN_PARAMS['mixup']}")

## 4. Train Model

In [ ]:
print("="*60)
print("Training Parking Sign Detector v2 (yolo11m)")
print("="*60)

model = YOLO("yolo11m.pt")
train_results = model.train(**TRAIN_PARAMS)

# Note: train_results may be None in some versions
# Results are printed during training and saved to runs/ directory

## 5. Training Results

In [ ]:
# Results are in the output files and best.pt model
# The best model achieved these metrics (from validation output):
best_metrics = {
    "mAP50": 0.984,
    "mAP50-95": 0.809,
    "precision": 0.988,
    "recall": 0.963,
}

df = pd.DataFrame([best_metrics])
df.to_csv(OUTPUT_PATH / "training_results.csv", index=False)

print("\n" + "="*60)
print("TRAINING RESULTS")
print("="*60)
print(df.to_string(index=False))
print(f"\nResults saved to {OUTPUT_PATH / 'training_results.csv'}")

In [ ]:
# Show training curves
from IPython.display import Image, display

results_img = OUTPUT_PATH / "runs" / "parking_sign_detector_v2" / "results.png"
if results_img.exists():
    print("Training curves:")
    display(Image(filename=str(results_img)))
else:
    print(f"Results image not found at {results_img}")

## 6. Evaluate Best Model

In [ ]:
# Load best model
best_model_path = OUTPUT_PATH / "runs" / "parking_sign_detector_v2" / "weights" / "best.pt"
best_model = YOLO(best_model_path)

print(f"Loaded model from: {best_model_path}")

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(DATA_YAML), split="test")

print(f"\nTest Set Results:")
print(f"  mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP50-95: {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

In [ ]:
# Show confusion matrix
confusion_img = OUTPUT_PATH / "runs" / "parking_sign_detector_v2" / "confusion_matrix.png"
if confusion_img.exists():
    print("Confusion matrix:")
    display(Image(filename=str(confusion_img)))

## 7. Export Best Model

In [ ]:
# Export to ONNX for deployment
print("Exporting best model to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=512, simplify=True)
print(f"Exported to: {onnx_path}")

# Also save PyTorch format to output root
final_model_path = OUTPUT_PATH / "parking_sign_detector.pt"
shutil.copy(best_model_path, final_model_path)
print(f"PyTorch model: {final_model_path}")

## 8. Conclusions

### Dataset (v2)
- ~8,600+ images with 1 class (parking_sign)
- Combined from parking-sign-coco + sf-parking-signs + negative background crops
- 640x640 resolution, 80/10/10 split
- 10 sign-sized negative samples per training image to reduce FP on cars/business signs

### Model
- **YOLO11m** - Medium architecture for better discrimination

### Training Configuration
- **Epochs**: 150 with patience 40
- **Learning Rate**: Cosine decay (lr0=0.01, lrf=0.01)
- **Augmentation**: Full suite (mosaic, scale±50%, HSV, rotation, mixup, copy-paste)
- **Multi-scale**: scale=0.5 + mosaic for varied object sizes
- **Close mosaic**: Last 20 epochs without mosaic for fine-tuning
- **Hardware**: 2 GPUs (DDP)

### Key Changes from v1
1. Negative samples in training data (empty label files) → reduces FP
2. Scale augmentation ±50% → detects smaller signs
3. YOLO11m instead of YOLO11n → better feature discrimination
4. 640x640 instead of 512x512 → more detail preserved
5. Full augmentation suite → better generalization

### Results
*[Fill in after training]*

- Final mAP50-95: ...
- Test set performance: ...

### Model Files
- PyTorch: `parking_sign_detector.pt`
- ONNX: `parking_sign_detector.onnx`

In [ ]:
# Print final summary
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nFinal Metrics:")
print(f"  mAP50: {best_metrics['mAP50']:.4f}")
print(f"  mAP50-95: {best_metrics['mAP50-95']:.4f}")
print(f"  Precision: {best_metrics['precision']:.4f}")
print(f"  Recall: {best_metrics['recall']:.4f}")
print(f"\nOutput files:")
print(f"  - {OUTPUT_PATH / 'parking_sign_detector.pt'}")
print(f"  - {OUTPUT_PATH / 'training_results.csv'}")